In [1]:
import tensorflow as tf

from tensorflow.keras import layers

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import GlobalAveragePooling2D

from tensorflow.keras.applications import MobileNetV2

import numpy as np

import cv2

import os

In [2]:
train_path = r"C:\Users\akash\Desktop\DEEP LEARNING\Car VS College Bus\Dataset\Train"

test_path = r"C:\Users\akash\Desktop\DEEP LEARNING\Car VS College Bus\Dataset\Test"

IMG_SIZE = 128

BATCH_SIZE = 32

In [3]:
bus_train = len(os.listdir(train_path + r"\bus"))

car_train = len(os.listdir(train_path + r"\car"))

print("Bus Images :", bus_train)

print("Car Images :", car_train)

print("Total Images :", bus_train + car_train)

Bus Images : 265
Car Images : 260
Total Images : 525


In [4]:
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(

    train_path,

    image_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    label_mode='binary'

)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(

    test_path,

    image_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    label_mode='binary'

)

Found 523 files belonging to 2 classes.


Found 117 files belonging to 2 classes.


In [5]:
class_names = train_dataset.class_names

print("Classes :", class_names)

Classes : ['bus', 'car']


In [6]:
normalization_layer = layers.Rescaling(1./255)

train_dataset = train_dataset.map(

    lambda x, y: (normalization_layer(x), y)

)

test_dataset = test_dataset.map(

    lambda x, y: (normalization_layer(x), y)

)

In [7]:
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.1),

    layers.RandomZoom(0.1)

])

In [8]:
base_model = MobileNetV2(

    weights='imagenet',

    include_top=False,

    input_shape=(IMG_SIZE, IMG_SIZE, 3)

)

base_model.trainable = False

In [9]:
model = Sequential([

    data_augmentation,

    base_model,

    GlobalAveragePooling2D(),

    Dense(128, activation='relu'),

    Dropout(0.5),

    Dense(1, activation='sigmoid')

])

In [10]:
model.build((None, IMG_SIZE, IMG_SIZE, 3))

In [10]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

In [11]:
model.compile(

    optimizer='adam',

    loss='binary_crossentropy',

    metrics=['accuracy']

)

In [12]:
history = model.fit(

    train_dataset,

    validation_data=test_dataset,

    epochs=2

)

Epoch 1/2


17/17 ━━━━━━━━━━━━━━━━━━━━ 17s 506ms/step - accuracy: 0.8967 - loss: 0.2344 - val_accuracy: 0.9915 - val_loss: 0.0342
Epoch 2/2
17/17 ━━━━━━━━━━━━━━━━━━━━ 7s 424ms/step - accuracy: 0.9656 - loss: 0.1024 - val_accuracy: 1.0000 - val_loss: 0.0209


In [13]:
model.save("college_bus_vs_car_model.h5")

print("Model Saved Successfully!")

Model Saved Successfully!


In [14]:
train_accuracy = history.history['accuracy'][-1]

val_accuracy = history.history['val_accuracy'][-1]

train_loss = history.history['loss'][-1]

val_loss = history.history['val_loss'][-1]

print("\n==============================")

print("MODEL PERFORMANCE")

print("==============================")

print(f"Training Accuracy    : {train_accuracy:.4f}")

print(f"Validation Accuracy  : {val_accuracy:.4f}")

print(f"Training Loss        : {train_loss:.4f}")

print(f"Validation Loss      : {val_loss:.4f}")

print("==============================")


MODEL PERFORMANCE
Training Accuracy    : 0.9656
Validation Accuracy  : 1.0000
Training Loss        : 0.1024
Validation Loss      : 0.0209


In [15]:
test_loss, test_accuracy = model.evaluate(test_dataset)

print("\n========================")

print("FINAL TEST PERFORMANCE")

print("========================")

print(f"Test Accuracy : {test_accuracy:.4f}")

print(f"Test Loss     : {test_loss:.4f}")

print("========================")

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 365ms/step - accuracy: 1.0000 - loss: 0.0209

FINAL TEST PERFORMANCE
Test Accuracy : 1.0000
Test Loss     : 0.0209


In [16]:
def predict_vehicle(image_path):
    
    img = cv2.imread(image_path)

    if img is None:

        print("Image not found!")

        return

    display_img = img.copy()

    # Resize Image
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # Normalize Image
    img = img.astype("float32") / 255.0

    # Expand Dimensions
    img = np.expand_dims(img, axis=0)

    # Prediction
    prediction = model.predict(img, verbose=0)

    score = prediction[0][0]

    # Classification
    if score >= 0.5:

        result = "CAR"

        confidence = score * 100

    else:

        result = "COLLEGE BUS"

        confidence = (1 - score) * 100

    print("\n==============================")

    print("VEHICLE IDENTIFICATION SYSTEM")

    print("==============================")

    print("Prediction Score :", score)

    print("Predicted Class  :", result)

    print(f"Confidence       : {confidence:.2f}%")

    print("==============================")

    # Display Result on Image
    cv2.putText(

        display_img,

        f"{result} ({confidence:.2f}%)",

        (20,40),

        cv2.FONT_HERSHEY_SIMPLEX,

        1,

        (0,255,0),

        2

    )

    cv2.imshow("Prediction", display_img)

    cv2.waitKey(0)

    cv2.destroyAllWindows()

In [17]:
def campus_vehicle_monitoring(folder_path):
    
    bus_count = 0

    car_count = 0

    print("\n===================================")

    print(" CAMPUS VEHICLE MONITORING SYSTEM ")

    print("===================================")

    for image_name in os.listdir(folder_path):

        image_path = os.path.join(folder_path, image_name)

        img = cv2.imread(image_path)

        if img is None:
            continue

        display_img = img.copy()

        # Resize Image
        img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        # Normalize Image
        img_array = img_resized.astype("float32") / 255.0

        # Expand Dimensions
        img_array = np.expand_dims(img_array, axis=0)

        # Prediction
        prediction = model.predict(img_array, verbose=0)

        score = prediction[0][0]

        # Classification
        if score >= 0.5:

            result = "CAR"

            car_count += 1

        else:

            result = "COLLEGE BUS"

            bus_count += 1

        # Display Prediction
        cv2.putText(

            display_img,

            result,

            (20,40),

            cv2.FONT_HERSHEY_SIMPLEX,

            1,

            (0,255,0),

            2

        )

        cv2.imshow("Campus Vehicle Monitoring", display_img)

        #print(f"{image_name} --> {result}")

        cv2.waitKey(1000)

    cv2.destroyAllWindows()

    print("\n===================================")

    print(" VEHICLE COUNTING SYSTEM ")

    print("===================================")

    print("Total College Buses :", bus_count)

    print("Total Cars          :", car_count)

    print("Total Vehicles      :", bus_count + car_count)

    print("===================================")